In [1]:
import pandas as pd
import numpy as np
import os
import time
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
train_path = "cyber-pack(4)/Cyber/training/flows.parquet"
eval_path = "cyber-pack(4)/Cyber/eval/flows.parquet"

train_df = pd.read_parquet(train_path)
eval_df = pd.read_parquet(eval_path)

print("Training shape:", train_df.shape)
print("Evaluation shape:", eval_df.shape)

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nEvaluation labels:")
print(eval_df["label"].value_counts())

FileNotFoundError: cyber-pack(4)/Cyber/training/flows.parquet

In [ ]:
TARGET = "label"
DROP_COLS = ["label", "record_id"]

assert TARGET in train_df.columns
assert TARGET in eval_df.columns

feature_cols = [
    col for col in train_df.columns
    if col not in DROP_COLS
]

assert set(feature_cols).issubset(eval_df.columns)

X_train = train_df[feature_cols].copy()
y_train = train_df[TARGET].copy()

X_eval = eval_df[feature_cols].copy()
y_eval = eval_df[TARGET].copy()

# Convert infinite values to missing values
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_eval = X_eval.replace([np.inf, -np.inf], np.nan)

# Ensure numeric features
assert all(pd.api.types.is_numeric_dtype(X_train[c]) for c in feature_cols)

print("Features:", len(feature_cols))
print("Training samples:", len(X_train))
print("Evaluation samples:", len(X_eval))
print("Labels:", sorted(y_train.unique()))

In [ ]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=30,
        max_depth=20,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

start = time.time()

model.fit(X_train, y_train)

training_time = time.time() - start

print(f"Training completed in {training_time:.2f} seconds!")

In [ ]:
y_pred = model.predict(X_eval)

accuracy = accuracy_score(y_eval, y_pred)
macro_f1 = f1_score(
    y_eval, y_pred,
    average="macro",
    zero_division=0
)

print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))

print("\nClassification Report:")
print(classification_report(y_eval, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_eval, y_pred))

In [ ]:
os.makedirs("models", exist_ok=True)

joblib.dump(
    {
        "model": model,
        "feature_columns": feature_cols,
        "classes": list(model.classes_)
    },
    "models/agent1_random_forest.joblib"
)

print("Model saved successfully!")